# NovaCloud Corp

# Proyecto RAG ~ Asistente SOC de Respuesta a Incidentes y Normativa 
# 01 ~ Base R0

**Sistemas Inteligentes Computacionales ~ Parcial Práctico Sustentable (Corte 2)**

## 1. Definición del Caso de Uso
* **Caso de aplicación:** Asistente de Mesa de Ayuda / Centro de Operaciones de Seguridad (SOC) para consultar políticas de acceso, playbooks de respuesta a incidentes, matriz de SLA/escalamiento, guía de hardening e historial de incidentes resueltos.
* **Usuarios previstos:** Analistas SOC Nivel 1 y Nivel 2 durante el triaje y contención de incidentes.
* **Preguntas que debe resolver:** Procedimientos de contención (phishing, ransomware, fuerza bruta), tiempos máximos de SLA y rutas de escalamiento, reglas de contraseñas/MFA/red y antecedentes de incidentes previos.
* **Fuera de alcance:** Consultas administrativas de RRHH, presupuestos financieros, soporte de software ofimático general o ejecución automática de comandos sobre infraestructura.
* **Naturaleza del corpus:** 5 documentos **sintéticos identificados como tales**, en 3 formatos distintos (`.pdf`, `.docx`, `.md`), alojados en la carpeta `corpus/`.

In [1]:
# Instalación de dependencias (ejecutar si hace falta en el entorno)
%pip install -q openai sentence-transformers pypdf python-docx rank_bm25

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import time
import numpy as np
from pypdf import PdfReader
from docx import Document
from openai import OpenAI
from sentence_transformers import SentenceTransformer

# ----------------------------------------------------------------------
# PARÁMETROS GLOBALES Y CONFIGURACIÓN BASE (R0)
# Todos agrupados para garantizar reproducibilidad y justificación técnica
# ----------------------------------------------------------------------
CARPETA_CORPUS = os.path.join("..", "corpus") if not os.path.isdir("corpus") and os.path.isdir(os.path.join("..", "corpus")) else "corpus"

# Modelos
MODELO_EMBEDDINGS_R0 = "paraphrase-multilingual-MiniLM-L12-v2"  # Límite: 128 tokens, optimizado para español
MODELO_LLM_R0 = "deepseek/deepseek-chat"                        # Vía OpenRouter

# Parámetros de fragmentación (R0)
# Nota: Usamos 400 caracteres (~85-100 tokens) y 60 caracteres de solapamiento (15%)
# para respetar el límite máximo de 128 tokens del modelo de embeddings sin truncar texto.
CHUNK_SIZE_R0 = 400      # caracteres
OVERLAP_R0 = 60          # caracteres (15% de solapamiento)

# Parámetros de recuperación y generación (fijos entre variantes salvo lo que cambie el experimento)
TOP_K_R0 = 4
TEMPERATURA = 0.0
MAX_TOKENS_SALIDA = 500
MAX_CONTEXTO_CHARS = 6000  # Límite de seguridad para el contexto enviado al LLM

# Cliente OpenRouter (la clave se lee desde variable de entorno, nunca hardcodeada)
api_key_openrouter = os.getenv("API_KEY") or os.getenv("OPENROUTER_API_KEY")
if api_key_openrouter:
    client = OpenAI(
        api_key=api_key_openrouter,
        base_url="https://openrouter.ai/api/v1",
        default_headers={
            "HTTP-Referer": "http://localhost",
            "X-Title": "Asistente SOC RAG - NovaCloud",
        },
    )
else:
    client = None
    print("AVISO: Define API_KEY o OPENROUTER_API_KEY en tu entorno antes de consultar el LLM.")

embedder_r0 = SentenceTransformer(MODELO_EMBEDDINGS_R0)
print(f"Embedder R0 cargado: {MODELO_EMBEDDINGS_R0} (max_seq_length={embedder_r0.max_seq_length} tokens)")

c:\Users\sebas\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4715.68it/s]


Embedder R0 cargado: paraphrase-multilingual-MiniLM-L12-v2 (max_seq_length=128 tokens)


## 2. Ingesta Multi-Formato con Trazabilidad
Cada documento en `corpus/` (`.pdf`, `.docx`, `.md`) se lee extrayendo sus unidades de texto junto con su **`archivo`** de origen y un **`localizador`** verificable (número de página, sección/párrafo o líneas).

In [3]:
def cargar_corpus(carpeta=CARPETA_CORPUS):
    """
    Lee todos los archivos .pdf, .docx y .md de la carpeta indicada.
    Retorna una lista de registros: {"archivo": str, "localizador": str, "seccion": str, "texto": str}
    """
    if not os.path.isdir(carpeta):
        raise FileNotFoundError(f"No existe la carpeta del corpus: {carpeta}")

    registros = []
    for nombre in sorted(os.listdir(carpeta)):
        ruta = os.path.join(carpeta, nombre)
        ext = os.path.splitext(nombre)[1].lower()

        if ext == ".pdf":
            reader = PdfReader(ruta)
            for num_pag, page in enumerate(reader.pages, start=1):
                texto = (page.extract_text() or "").strip()
                if texto:
                    # Limpiar saltos de línea excesivos manteniendo párrafos
                    texto_limpio = re.sub(r"\s+", " ", texto).strip()
                    registros.append({
                        "archivo": nombre,
                        "localizador": f"Página {num_pag}",
                        "seccion": f"Página {num_pag}",
                        "texto": texto_limpio,
                    })

        elif ext == ".docx":
            doc = Document(ruta)
            seccion_actual = "Inicio"
            for idx, p in enumerate(doc.paragraphs, start=1):
                txt = p.text.strip()
                if not txt:
                    continue
                if p.style and p.style.name.startswith("Heading"):
                    seccion_actual = txt
                registros.append({
                    "archivo": nombre,
                    "localizador": f"Sección '{seccion_actual}' (Párrafo {idx})",
                    "seccion": seccion_actual,
                    "texto": txt,
                })
            for t_idx, tabla in enumerate(doc.tables, start=1):
                filas_txt = []
                for row in tabla.rows:
                    celdas = [c.text.strip() for c in row.cells if c.text.strip()]
                    if celdas:
                        filas_txt.append(" | ".join(celdas))
                if filas_txt:
                    registros.append({
                        "archivo": nombre,
                        "localizador": f"Tabla {t_idx}",
                        "seccion": f"Tabla {t_idx}",
                        "texto": "\n".join(filas_txt),
                    })

        elif ext == ".md":
            with open(ruta, "r", encoding="utf-8") as f:
                lineas = f.readlines()
            seccion_actual = "Inicio"
            bloque, inicio_linea = [], 1
            for num_linea, linea in enumerate(lineas, start=1):
                limpia = linea.strip()
                if limpia.startswith("#"):
                    if bloque:
                        registros.append({
                            "archivo": nombre,
                            "localizador": f"Sección '{seccion_actual}' (Líneas {inicio_linea}-{num_linea - 1})",
                            "seccion": seccion_actual,
                            "texto": " ".join(bloque),
                        })
                        bloque = []
                    seccion_actual = limpia.lstrip("#").strip()
                    inicio_linea = num_linea
                    bloque.append(seccion_actual)
                elif limpia and set(limpia) != {"-"}:
                    if not bloque:
                        inicio_linea = num_linea
                    bloque.append(limpia)
            if bloque:
                registros.append({
                    "archivo": nombre,
                    "localizador": f"Sección '{seccion_actual}' (Líneas {inicio_linea}-{len(lineas)})",
                    "seccion": seccion_actual,
                    "texto": " ".join(bloque),
                })

    return registros


unidades_corpus = cargar_corpus()
archivos_unicos = sorted(set(u["archivo"] for u in unidades_corpus))
print(f"Archivos cargados ({len(archivos_unicos)}): {archivos_unicos}")
print(f"Unidades estructurales extraídas: {len(unidades_corpus)}")

Archivos cargados (5): ['Guia_Hardening_Red_y_Servidores.pdf', 'Matriz_SLA_y_Escalamiento.docx', 'Playbook_Respuesta_Incidentes.md', 'Politica_Seguridad_y_Accesos.pdf', 'Registro_Incidentes_Resueltos.md']
Unidades estructurales extraídas: 31


## 3. Fragmentación (Chunking) e Indexación ~ Configuración Base R0
Dividimos cada unidad en fragmentos de tamaño fijo (`CHUNK_SIZE_R0 = 400` caracteres) con solapamiento (`OVERLAP_R0 = 60` caracteres), conservando siempre `archivo` y `localizador`.

In [4]:
def fragmentar_tamano_fijo(unidades, chunk_size=CHUNK_SIZE_R0, overlap=OVERLAP_R0):
    """
    Fragmenta las unidades del corpus en ventanas de tamaño fijo (caracteres) con solapamiento,
    respetando los límites de oración cuando es posible y conservando archivo y localizador.
    """
    chunks = []
    paso = max(1, chunk_size - overlap)
    chunk_id = 0

    for u in unidades:
        texto = u["texto"]
        if len(texto) <= chunk_size:
            chunks.append({
                "id": chunk_id,
                "archivo": u["archivo"],
                "localizador": u["localizador"],
                "texto": texto,
            })
            chunk_id += 1
        else:
            inicio = 0
            parte = 1
            while inicio < len(texto):
                fin = min(len(texto), inicio + chunk_size)
                fragmento = texto[inicio:fin].strip()
                if fragmento:
                    chunks.append({
                        "id": chunk_id,
                        "archivo": u["archivo"],
                        "localizador": f"{u['localizador']} [Parte {parte}]",
                        "texto": fragmento,
                    })
                    chunk_id += 1
                    parte += 1
                if fin == len(texto):
                    break
                inicio += paso
    return chunks


def diagnosticar_tokens(chunks, embedder):
    limite = embedder.max_seq_length
    largos = [len(embedder.tokenizer.tokenize(c["texto"])) for c in chunks]
    excedidos = sum(1 for l in largos if l > limite)
    print(f"Total chunks: {len(chunks)} | Promedio tokens: {np.mean(largos):.1f} | "
          f"Máx tokens: {max(largos)} | Límite embedder: {limite} | Excedidos: {excedidos}")


chunks_r0 = fragmentar_tamano_fijo(unidades_corpus, CHUNK_SIZE_R0, OVERLAP_R0)
diagnosticar_tokens(chunks_r0, embedder_r0)

embeddings_r0 = embedder_r0.encode([c["texto"] for c in chunks_r0], normalize_embeddings=True)
print(f"Índice R0 construido: {embeddings_r0.shape}")

Total chunks: 56 | Promedio tokens: 74.7 | Máx tokens: 126 | Límite embedder: 128 | Excedidos: 0
Índice R0 construido: (56, 384)


## 4. Recuperación, Plantilla de Respuesta con Citas y Abstención (R0)

In [5]:
PLANTILLA_SISTEMA_RAG = (
    "Eres un asistente técnico del Centro de Operaciones de Seguridad (SOC) de NovaCloud Corp. "
    "Responde únicamente con base en los fragmentos de contexto provistos. "
    "Cada afirmación debe incluir su cita exacta entre corchetes con el formato [Archivo | Localizador]. "
    "Si el contexto no contiene información suficiente para responder la pregunta de forma completa y verificable, "
    "debes abstenerte respondiendo exactamente: 'No encuentro evidencia suficiente en los documentos del corpus para responder esta pregunta.' "
    "No uses conocimiento externo ni inventes datos."
)


def recuperar_denso(pregunta, chunks, chunk_embeddings, embedder, top_k=TOP_K_R0):
    query_emb = embedder.encode(pregunta, normalize_embeddings=True)
    similitudes = chunk_embeddings @ query_emb
    top_indices = np.argsort(similitudes)[::-1][:top_k]
    return [
        {
            "rank": rank + 1,
            "score": float(similitudes[i]),
            "archivo": chunks[i]["archivo"],
            "localizador": chunks[i]["localizador"],
            "texto": chunks[i]["texto"],
        }
        for rank, i in enumerate(top_indices)
    ]


def construir_contexto(recuperados, max_chars=MAX_CONTEXTO_CHARS):
    bloques = []
    acumulado = 0
    recortado = False
    for r in recuperados:
        bloque = f"[{r['archivo']} | {r['localizador']}] (score={r['score']:.3f})\n{r['texto']}"
        if acumulado + len(bloque) > max_chars:
            recortado = True
            break
        bloques.append(bloque)
        acumulado += len(bloque) + 2
    return "\n\n".join(bloques), recortado


def consultar_rag(pregunta, chunks, chunk_embeddings, embedder=embedder_r0,
                  modelo_llm=MODELO_LLM_R0, top_k=TOP_K_R0):
    t0 = time.perf_counter()
    recuperados = recuperar_denso(pregunta, chunks, chunk_embeddings, embedder, top_k=top_k)
    contexto, recortado = construir_contexto(recuperados)

    prompt_usuario = f"""Contexto recuperado:
{contexto}

Pregunta: {pregunta}

Instrucción: Responde en español, de forma precisa, citando [Archivo | Localizador] en cada afirmación."""

    response = client.chat.completions.create(
        model=modelo_llm,
        messages=[
            {"role": "system", "content": PLANTILLA_SISTEMA_RAG},
            {"role": "user", "content": prompt_usuario},
        ],
        temperature=TEMPERATURA,
        max_tokens=MAX_TOKENS_SALIDA,
    )
    latencia = time.perf_counter() - t0
    return {
        "pregunta": pregunta,
        "respuesta": response.choices[0].message.content,
        "recuperados": recuperados,
        "contexto_recortado": recortado,
        "tiempo_seg": round(latencia, 2),
    }

## 5. Prueba Rápida de Desarrollo (Validación R0)
Probamos el flujo base con 3 preguntas de desarrollo (1 directa, 1 combinada y 1 fuera de alcance) antes de pasar a la Fase 3 (variantes experimentales).

In [6]:
PRUEBAS_DESARROLLO = [
    ("Directa", "¿En qué puerto debe escuchar el servicio SSH y qué métodos de autenticación están permitidos?"),
    ("Multi-documento", "Si se detecta sospecha de ransomware en un equipo, ¿qué debe hacerse con el equipo según el playbook y cuál es el tiempo máximo de primera respuesta y notificación al CISO?"),
    ("Sin respuesta", "¿Cuál es el presupuesto anual asignado al SOC de NovaCloud Corp para licencias de software?"),
]

for tipo, preg in PRUEBAS_DESARROLLO:
    res = consultar_rag(preg, chunks_r0, embeddings_r0)
    print("=" * 80)
    print(f"[{tipo}] ({res['tiempo_seg']}s | recortado={res['contexto_recortado']})")
    print(f"Pregunta: {preg}\n")
    print("Fragmentos recuperados:")
    for r in res["recuperados"]:
        print(f"  #{r['rank']} ({r['score']:.3f}) [{r['archivo']} | {r['localizador']}]: {r['texto'][:100]}...")
    print(f"\nRespuesta:\n{res['respuesta']}\n")

[Directa] (3.22s | recortado=False)
Pregunta: ¿En qué puerto debe escuchar el servicio SSH y qué métodos de autenticación están permitidos?

Fragmentos recuperados:
  #1 (0.773) [Guia_Hardening_Red_y_Servidores.pdf | Página 1 [Parte 2]]: prohíbe exponer el puerto estándar 22. El servicio SSH debe escuchar en el puerto 2222.  Acceso de ...
  #2 (0.666) [Guia_Hardening_Red_y_Servidores.pdf | Página 1 [Parte 1]]: Guía Técnica de Hardening de Red y Servidores (GUI-INF-04) Organización: NovaCloud Corp | Área: Infr...
  #3 (0.606) [Registro_Incidentes_Resueltos.md | Sección 'INC-2026-019 — Ataque de fuerza bruta contra bastión SSH' (Líneas 17-26) [Parte 1]]: INC-2026-019 — Ataque de fuerza bruta contra bastión SSH * **Fecha:** 2026-09-03 * **Severidad:** P3...
  #4 (0.572) [Guia_Hardening_Red_y_Servidores.pdf | Página 1 [Parte 4]]: os los siguientes puertos: Zona / Subred Puertos Permitidos Origen Autorizado Notas DMZ / Balanceado...

Respuesta:
El servicio SSH debe escuchar en el puerto 22